# SecurAI — Init Colab (SSH + tunnel Cloudflare)

Runtime → **GPU T4**. Puis depuis Windows : `ssh vision-colab`

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

SECURAI_BASE = '/content/drive/MyDrive/Vision_project/securai_store'
CF_TOKEN     = 'COLLE_TON_TOKEN_CLOUDFLARE_ICI'
SSH_PASSWORD = 'MotDePasseTemp2024!'

import os
os.environ['SECURAI_BASE'] = SECURAI_BASE
for folder in ['checkpoints', 'output', 'logs', 'data/enrolled', 'models']:
    os.makedirs(f'{SECURAI_BASE}/{folder}', exist_ok=True)

print('✓ Drive monté —', SECURAI_BASE)

In [ ]:
import subprocess, os

for cmd in [
    'apt-get update -qq',
    'apt-get install -y -qq openssh-server',
    'mkdir -p /var/run/sshd',
]:
    subprocess.run(cmd, shell=True, check=True)

with open('/etc/ssh/sshd_config', 'a') as f:
    f.write('\nPermitRootLogin yes\nPasswordAuthentication yes\n')

os.system(f'echo "root:{SSH_PASSWORD}" | chpasswd')
print('✓ SSH configuré.')

In [ ]:
import subprocess, time

subprocess.run(
    'wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/'
    'cloudflared-linux-amd64 -O /usr/local/bin/cloudflared && chmod +x /usr/local/bin/cloudflared',
    shell=True, check=True
)

subprocess.run('service ssh start', shell=True)

log_path = f'{SECURAI_BASE}/logs/tunnel.log'
proc = subprocess.Popen(
    f'cloudflared tunnel --no-autoupdate run --token {CF_TOKEN}',
    shell=True,
    stdout=open(log_path, 'w'),
    stderr=subprocess.STDOUT
)

time.sleep(6)
print(f'✓ Tunnel actif (PID: {proc.pid})')
print('→ Connexion Windows : ssh vision-colab')